# Initiliaze

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim,col

# Read Bronze table "crm_cust_info"

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")
display(df)

# Silver Transformations

##Trimming
parse all df and trim it col by col.

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim((col(field.name))))

##Nulls
handle or delete unnecessary nulls from cst_id


In [0]:
df = df.filter(col("cst_id").isNotNull())


##Normalizations | Abbriviations
Encrich data in tables cst_marital_stauts and cst_gndr

In [0]:
df = (
    df
    .withColumn(
        "cst_gndr",
        F.when(F.lower(F.col("cst_gndr")).isin("m", "male"), "Male")
         .when(F.lower(F.col("cst_gndr")).isin("f", "female"), "Female")
         .otherwise("n/a")
    )
    .withColumn(
        "cst_marital_status",
         F.when(F.lower(F.col("cst_marital_status")).isin("s", "single"), "Single")
          .when(F.lower(F.col("cst_marital_status")).isin("m", "married"), "Married")
          .otherwise("n/a")
        ))


##Rename Collumns
Give friendly names for your Collumns

In [0]:
RENAME_COLLUMNS = {
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "firstname",
    "cst_lastname": "lastname",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}

for old_name, new_name in RENAME_COLLUMNS.items():
    df = df.withColumnRenamed(old_name, new_name)

## Sanity checks of dataframe


In [0]:
df.limit(10).display()

# Write Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_customers")


## Sanity checks for Silver Table

In [0]:
%sql
SELECT  * 
FROM workspace.silver.crm_customers LIMIT 10